In [72]:
"""
====================================================
Download ERA5 Data
====================================================
"""

'\n====================================================\nDownload ERA5 Data\n====================================================\n'

In [73]:
#######################
# DIRECTORIES

In [74]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
dataDirectory=mainDirectory+"../DATA/ERA5_Data/"
import os; os.makedirs(dataDirectory, exist_ok=True)

In [75]:
#######################
# LIBRARIES, FUNCTIONS, and CLASSES

In [76]:
# IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "/Libraries/"
sys.path.append(path)

# --- Import all your function modules ---
import importlib

modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [77]:
# IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [78]:
# IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/Classes/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "Classes_1",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [79]:
###########################
# DOWNLOADING DATA FUNCTIONS

In [80]:
# DOWNLOADING ERA5

# "Download ERA data with python" Code Inspired by https://github.com/joaohenry23/Download_ERA5_with_python
#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-pressure-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                "pressure_level": ["100", "250", "500", "750", "1000"],
                "date": date,
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area,
                "grid": [0.25, 0.25],
            },
            os.path.join(
                dataDirectory, date_folder, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_V2(variables,date_string_converted, area)

In [81]:
# DATE INFORMATION
def date_string_to_range(date_string: str) -> str:
    """
    Convert a date string like "06-30 - 07-02 (2022)"
    into ERA5 API format: "2022-06-30/to/2022-07-02".
    """
    # Extract year
    year = date_string.split("(")[1].replace(")", "").strip()

    # Extract the two parts safely
    date_part = date_string.split("(")[0].strip()  # "06-30 - 07-02"
    start, end = date_part.split(" - ")            # ["06-30", "07-02"]

    # Make full YYYY-MM-DD
    start_date = f"{year}-{start}"
    end_date   = f"{year}-{end}"

    return f"{start_date}/{end_date}"

    
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    # adding date to output folder
    subdir = os.path.join(dataDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder


# COORDINATES INFORMATION
def GetCoordinates(longitude, latitude, dx_m=250e3, dy_m=250e3, grid_res=0.25):
    longitude = coordinates.DMSToDecimal(*longitude)
    latitude = coordinates.DMSToDecimal(*latitude)

    dlon = coordinates.dxTOdlon(dx_m=dx_m, lat_deg=latitude)
    dlat = coordinates.dyTOdlat(dy_m=dy_m)

    N, W, S, E = [latitude + dlat, longitude - dlon, latitude - dlat, longitude + dlon]
    print("Coords box:", [N, W, S, E])
    # Round outward to 0.25 grid
    N = math.ceil(N / grid_res) * grid_res  # round north up
    S = math.floor(S / grid_res) * grid_res  # round south down
    W = math.floor(W / grid_res) * grid_res  # round west down (more negative)
    E = math.ceil(E / grid_res) * grid_res  # round east up

    area = [N, W, S, E]
    print("Rounded box:", area)
    return area


# VARIABLES INFORMATION
def GetVariableNames():
    variables = [
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "divergence",
        "vorticity",
        "temperature",
        "specific_humidity",
        "specific_cloud_liquid_water_content",
        "specific_cloud_ice_water_content",
        "specific_rain_water_content",
        "relative_humidity",
        "cloud_cover",
        "geopotential",
    ]
    return variables

In [82]:
###########################
# DOWNLOADING TRACER DATA

In [83]:
# coorindates information
# GETTING BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# 29°31'55"N, 95°17'2"W
longitude = (95, 17, 2, "W")
latitude = (29, 31, 55, "N")
area = GetCoordinates(longitude, latitude)
variables = GetVariableNames()

Coords box: [31.78024845924127, -97.86790574593715, 27.28364042964762, -92.69987203184061]
Rounded box: [32.0, -98.0, 27.25, -92.5]


In [84]:
###########################
# DATE ONE (BORING CASE)

In [85]:
# INFORMATION
# date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 09:21:17,968 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 09:21:18,412 INFO Request ID is 89bcb1fd-c1fe-4b28-bdfd-f8a4282eeecd
2025-09-04 09:21:18,583 INFO status has been updated to accepted
2025-09-04 09:21:41,015 INFO status has been updated to running
2025-09-04 09:22:09,837 INFO status has been updated to successful


cf35f7392cb26724152e9fd1b985d5b2.nc:   0%|          | 0.00/382k [00:00<?, ?B/s]

2025-09-04 09:22:12,135 INFO Request ID is 7db68906-1887-435c-a34b-f57c1e9c591f
2025-09-04 09:22:12,293 INFO status has been updated to accepted
2025-09-04 09:22:34,306 INFO status has been updated to running
2025-09-04 09:23:03,131 INFO status has been updated to successful


2f52591c39165717bb022961c3b22788.nc:   0%|          | 0.00/388k [00:00<?, ?B/s]

...

d59b8267a063fd7e8d2f9a9c1bafdf2c.nc:   0%|          | 0.00/376k [00:00<?, ?B/s]

2025-09-04 09:33:53,357 INFO Request ID is 677244e7-0a58-40f0-82ec-965f283b2455
2025-09-04 09:33:53,632 INFO status has been updated to accepted
2025-09-04 09:34:02,588 INFO status has been updated to running
2025-09-04 09:34:44,699 INFO status has been updated to successful


ed007d7813520b7bbbec26a71416e211.nc:   0%|          | 0.00/61.3k [00:00<?, ?B/s]

2025-09-04 09:34:46,624 INFO Request ID is 78444149-2051-4a18-9b55-2e608237ea79
2025-09-04 09:34:46,772 INFO status has been updated to accepted
2025-09-04 09:35:00,814 INFO status has been updated to running
2025-09-04 09:35:37,463 INFO status has been updated to successful


1806c8f5f37e6e199cc9c628981160fe.nc:   0%|          | 0.00/261k [00:00<?, ?B/s]

In [86]:
###########################
# DATE TWO (RAINY CASE)

In [87]:
# INFORMATION
# date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 09:35:40,992 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 09:35:41,370 INFO Request ID is 447673f3-5a86-41cc-92b9-0debcd4cbaf1
2025-09-04 09:35:41,528 INFO status has been updated to accepted
2025-09-04 09:35:55,620 INFO status has been updated to successful


2af056123a333eb564b50a93b56245db.nc:   0%|          | 0.00/404k [00:00<?, ?B/s]

2025-09-04 09:35:57,877 INFO Request ID is 629a7680-bc27-4927-a925-004aee20d318
2025-09-04 09:35:58,037 INFO status has been updated to accepted
2025-09-04 09:36:06,906 INFO status has been updated to running
2025-09-04 09:36:12,143 INFO status has been updated to successful


7f799beeb525361d0588b99c1c0cb357.nc:   0%|          | 0.00/403k [00:00<?, ?B/s]

...

36ef7cd4502c9c36f6555a94f379db9f.nc:   0%|          | 0.00/374k [00:00<?, ?B/s]

2025-09-04 09:44:42,236 INFO Request ID is 4f2a9251-1413-408b-9bca-068ed5cf5532
2025-09-04 09:44:42,416 INFO status has been updated to accepted
2025-09-04 09:44:51,280 INFO status has been updated to running
2025-09-04 09:44:56,516 INFO status has been updated to accepted
2025-09-04 09:45:04,291 INFO status has been updated to running
2025-09-04 09:45:33,690 INFO status has been updated to successful


1381395fa134c5b599923da48874c772.nc:   0%|          | 0.00/111k [00:00<?, ?B/s]

2025-09-04 09:45:35,705 INFO Request ID is eb896e2f-9ed6-465c-8a04-5d86a9518069
2025-09-04 09:45:35,886 INFO status has been updated to accepted
2025-09-04 09:45:50,016 INFO status has been updated to running
2025-09-04 09:46:52,470 INFO status has been updated to successful


b10d81cbcf63817fcd1ddd7a57828a79.nc:   0%|          | 0.00/267k [00:00<?, ?B/s]

In [88]:
###########################
# DATE THREE (INTERESTING CASE)

In [89]:
# INFORMATION
# date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5(variables,date_string_converted, area)

2025-09-04 09:46:55,026 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-04 09:46:55,387 INFO Request ID is 39615c80-dc92-4a02-8bfb-95df34dc5033
2025-09-04 09:46:55,557 INFO status has been updated to accepted
2025-09-04 09:47:09,609 INFO status has been updated to running
2025-09-04 09:47:46,180 INFO status has been updated to successful


aea2362a7a00392a294fc98e4ba8a228.nc:   0%|          | 0.00/398k [00:00<?, ?B/s]

2025-09-04 09:47:49,010 INFO Request ID is 251f0050-2e0a-496e-bd5a-568e5adcb723
2025-09-04 09:47:49,160 INFO status has been updated to accepted
2025-09-04 09:48:03,151 INFO status has been updated to running
2025-09-04 09:48:39,926 INFO status has been updated to successful


2ff7736110ca40ea62a700b1c7a8df72.nc:   0%|          | 0.00/391k [00:00<?, ?B/s]

...

ecdd9ca935e0841430a62eae7f05e163.nc:   0%|          | 0.00/367k [00:00<?, ?B/s]

2025-09-04 09:57:33,999 INFO Request ID is 459b96e4-c6aa-476d-b468-25781e6b8a18
2025-09-04 09:57:34,165 INFO status has been updated to accepted
2025-09-04 09:57:48,375 INFO status has been updated to running
2025-09-04 09:58:25,196 INFO status has been updated to successful


acb7418bfbc2f8be3628921e73f7f023.nc:   0%|          | 0.00/117k [00:00<?, ?B/s]

2025-09-04 09:58:27,357 INFO Request ID is 7a9e239d-19ed-4664-b970-6ca922c87c86
2025-09-04 09:58:27,520 INFO status has been updated to accepted
2025-09-04 09:58:36,309 INFO status has been updated to running
2025-09-04 09:59:18,101 INFO status has been updated to successful


70bc8b6f2aa5c5df7d0fe2dee928af24.nc:   0%|          | 0.00/261k [00:00<?, ?B/s]